# VANGROVE #1 - Data Preparation & Balancing

VANGROVE (Visual Analytics & Navigation for Geographic Regional Output & Variety Evaluation)

### Deskripsi Notebook
Notebook ini berfokus pada tahap persiapan data (Data Preparation) untuk pengolahan dataset citra daun tanaman, yang meliputi proses penggabungan dataset, pembersihan data, serta penyeimbangan distribusi data sebelum digunakan pada tahap modeling. Tahapan yang dilakukan pada notebook ini meliputi:
1. **Data Gathering & Standardization**  
   Menggabungkan dataset daun tanaman (Jagung, Tomat, Mangga, dan Kentang) dari berbagai sumber serta melakukan standarisasi nama kelas penyakit agar konsisten

2. **Data Assessing & Cleaning**  
   Mendeteksi dan menghapus data duplikat menggunakan metode *MD5 hashing*, serta memvalidasi file gambar

3. **Dataset Balancing with Random Sampling**  
   Melakukan random sampling untuk menghasilkan distribusi data yang lebih seimbang antar kelas penyakit

**Output:**
Tahap preprocessing menghasilkan dataset gabungan (`combined`) yang telah melalui proses standarisasi dan pembersihan data untuk kebutuhan analisis lanjutan, serta dataset balanced (`balanced_700` dan `balanced_500`) yang digunakan pada tahap pemodelan dan pelatihan model CNN

In [ ]:
import os
import shutil
import hashlib
from PIL import Image
import pandas as pd
from collections import Counter
import random

In [2]:
IN_COLAB = 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive #type: ignore
    drive.mount('/content/drive')

## Data Gathering
Dataset dikumpulkan dari beberapa sumber dan digabungkan ke dalam satu struktur folder berdasarkan jenis tanaman dan penyakit

In [3]:
if IN_COLAB:
    print("Running on Google Colab")
    BASE_PATH = "/content/drive/MyDrive/VANGROVE/data"
else:
    print("Running on Local")
    BASE_PATH = "data"

RAW_PATH = os.path.join(BASE_PATH, "raw")
COMBINED_PATH = os.path.join(BASE_PATH, "combined")

os.makedirs(COMBINED_PATH, exist_ok=True)

Running on Local


In [10]:
dataset_mapping = {
    "corn-leaf-disease": "corn",
    "corn-or-maize-leaf-disease-dataset": "corn",
    "mango-leaf-DS-2025": "mango",
    "potato-leaf-disease-dataset": "potato",
    "tomato-leaves-dataset": "tomato"
}

Dataset dari berbagai sumber dipetakan ke dalam kategori tanaman: corn; mango; potato; tomato

In [11]:
def clean_name(name):
    return name.lower().replace(" ", "_").replace("-", "_")

In [12]:
corn_mapping = {
    "daun_sehat": "healthy",
    "healthy": "healthy",

    "karat_daun": "rust",
    "common_rust": "rust",

    "hawar_daun": "blight",
    "blight": "blight",

    "bercak_daun": "leaf_spot",
    "gray_leaf_spot": "leaf_spot"
}

In [13]:
def short_name(filename):
    name, ext = os.path.splitext(filename)
    return hashlib.md5(name.encode()).hexdigest() + ext.lower()

Dilakukan standardisasi nama folder dan file untuk menghindari inkonsistensi data

In [14]:
def process_folder(source_path, plant_name):
    file_count = 0
    error_count = 0

    mapping = corn_mapping if plant_name == "corn" else {}

    for disease in os.listdir(source_path):
        disease_path = os.path.join(source_path, disease)

        if not os.path.isdir(disease_path):
            continue

        clean_disease = clean_name(disease)
        final_disease = mapping.get(clean_disease, clean_disease)

        dest_folder = os.path.join(COMBINED_PATH, plant_name, final_disease)
        os.makedirs(dest_folder, exist_ok=True)

        for file in os.listdir(disease_path):
            src_file = os.path.join(disease_path, file)

            if not os.path.isfile(src_file):
                continue

            try:
                new_name = short_name(file)
                dst_file = os.path.join(dest_folder, new_name)

                # copy file
                if not os.path.exists(dst_file):
                    shutil.copy(src_file, dst_file)
                    file_count += 1

            except Exception as e:
                error_count += 1
                print(f"skip: {file}")

    return file_count, error_count

Menggabungkan seluruh dataset ke dalam folder combined dengan struktur: plant -> disease -> images

In [15]:
# reset
shutil.rmtree(COMBINED_PATH, ignore_errors=True)
os.makedirs(COMBINED_PATH, exist_ok=True)

# melanjutkan run
for dataset_name, plant_name in dataset_mapping.items():
    dataset_path = os.path.join(RAW_PATH, dataset_name)

    if not os.path.exists(dataset_path):
        continue

    subfolders = os.listdir(dataset_path)

    if "train" in subfolders or "valid" in subfolders:
        for split in ["train", "valid"]:
            split_path = os.path.join(dataset_path, split)
            if os.path.exists(split_path):
                files, errors = process_folder(split_path, plant_name)
    else:
        files, errors = process_folder(dataset_path, plant_name)

Folder combined selalu di-reset setiap kali proses dijalankan ulang. Hal ini dilakukan agar tahap assessing dilakukan pada data yang belum dibersihkan (masih mengandung duplikat dan data rusak)

## Data Assessing
Dilakukan evaluasi terhadap kualitas dataset, meliputi:
- Deteksi data duplikat
- Identifikasi gambar rusak
- Pemeriksaan format file

In [16]:
file_paths = []

for plant in os.listdir(COMBINED_PATH):
    plant_path = os.path.join(COMBINED_PATH, plant)

    for disease in os.listdir(plant_path):
        disease_path = os.path.join(plant_path, disease)

        for img in os.listdir(disease_path):
            img_path = os.path.join(disease_path, img)

            file_paths.append((plant, disease, img_path))

In [17]:
def get_image_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = {}
duplicates = []

for plant, disease, path in file_paths:
    try:
        img_hash = get_image_hash(path)

        if img_hash in hashes:
            duplicates.append(path)
        else:
            hashes[img_hash] = path

    except:
        pass

print("Jumlah duplikat:", len(duplicates))

Jumlah duplikat: 2277


In [18]:
bad_images = []

for _, _, path in file_paths:
    try:
        Image.open(path).verify()
    except:
        bad_images.append(path)

print("Gambar rusak:", len(bad_images))

Gambar rusak: 0


In [19]:
exts = set()

for _, _, path in file_paths:
    exts.add(path.split('.')[-1])

print(exts)

{'jpg', 'png', 'jpeg'}


### Insight
- Ditemukan data duplikat yang kemungkinan berasal dari penggabungan dataset berbeda
- Terdapat beberapa gambar yang tidak valid (corrupt)
- Format file bervariasi (jpg, jpeg, png)

## Data Cleaning
Dilakukan pembersihan dataset dengan:
- Menghapus data duplikat
- Menghapus gambar rusak
- Analisis distribusi data

In [20]:
for path in duplicates:
    os.remove(path)

print("Duplikat terhapus:", len(duplicates))

Duplikat terhapus: 2277


In [21]:
file_paths = []

for plant in os.listdir(COMBINED_PATH):
    for disease in os.listdir(os.path.join(COMBINED_PATH, plant)):
        for img in os.listdir(os.path.join(COMBINED_PATH, plant, disease)):
            file_paths.append((plant, disease, os.path.join(COMBINED_PATH, plant, disease, img)))

In [22]:
print("Total gambar:", len(file_paths))

Total gambar: 47516


In [23]:
counter = Counter()

for plant, disease, path in file_paths:
    counter[(plant, disease)] += 1

df_count = pd.DataFrame(
    [(p, d, c) for (p, d), c in counter.items()],
    columns=["plant", "disease", "count"]
)

df_count

,plant,disease,count
0,corn,blight,2146
1,corn,healthy,2162
2,corn,leaf_spot,998
3,corn,rust,2300
4,mango,anthrecnose,810
5,mango,dieback,972
6,mango,gall_mildge_damage,621
7,mango,healthy,1383
8,mango,insect_damage_webber,981
9,mango,leaf_blight,740


Insight
- Data duplikat berhasil dihapus sehingga mengurangi potensi bias
- Distribusi data belum sepenuhnya seimbang untuk, terutama pada dataset potato

## Dataset Balancing

Random sampling dilakukan untuk menyeimbangkan jumlah data pada setiap kelas dengan cara mengambil sejumlah gambar secara acak dari dataset asli

Dibuat dua versi dataset balanced:
- Dataset balanced_700 dengan maksimal 700 gambar per kelas
- Dataset balanced_500 dengan maksimal 500 gambar per kelas

Jika jumlah data pada suatu kelas melebihi target, maka data dipilih secara acak menggunakan random sampling. Sebaliknya, jika jumlah data kurang dari target, seluruh data tetap digunakan tanpa pengurangan

In [ ]:
random.seed(42)

SOURCE_DATASET = r"data\combined"
BALANCED_DATASET = r"data\balanced_700"

# target per tanaman
TARGETS = {
    "corn": 700,
    "mango": 700,
    "potato": 700,
    "tomato": 700
}

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")

for plant in os.listdir(SOURCE_DATASET):

    plant_path = os.path.join(SOURCE_DATASET, plant)

    if not os.path.isdir(plant_path):
        continue

    target_count = TARGETS.get(plant)

    print(f"\nProcessing {plant} (target = {target_count})")

    for disease in os.listdir(plant_path):

        disease_path = os.path.join(plant_path, disease)

        if not os.path.isdir(disease_path):
            continue

        output_folder = os.path.join(
            BALANCED_DATASET,
            plant,
            disease
        )

        os.makedirs(output_folder, exist_ok=True)
        
        files = [
            f for f in os.listdir(disease_path)
            if f.lower().endswith(VALID_EXTENSIONS)
        ]

        files.sort()

        original_count = len(files)

        # kalau lebih besar maka akan random sampling
        if original_count > target_count:

            selected_files = random.sample(files, target_count)

        # kalau lebih kecil copy semua
        else:

            selected_files = files

        # copy file
        for file in selected_files:

            src = os.path.join(disease_path, file)
            dst = os.path.join(output_folder, file)

            shutil.copy(src, dst)

        print(
            f"  {disease}: "
            f"{original_count} → {len(selected_files)}"
        )


Processing corn (target = 700)
  blight: 2146 → 700
  healthy: 2162 → 700
  leaf_spot: 998 → 700
  rust: 2300 → 700

Processing mango (target = 700)
  anthrecnose: 810 → 700
  dieback: 972 → 700
  gall_mildge_damage: 621 → 621
  healthy: 1383 → 700
  insect_damage_webber: 981 → 700
  leaf_blight: 740 → 700

Processing potato (target = 700)
  bacteria: 569 → 569
  fungi: 743 → 700
  healthy: 200 → 200
  pest: 603 → 603
  phytopthora: 309 → 309
  virus: 530 → 530

Processing tomato (target = 700)
  bacterial_spot: 3558 → 700
  early_blight: 2746 → 700
  healthy: 3706 → 700
  late_blight: 3732 → 700
  leaf_mold: 3094 → 700
  powdery_mildew: 1256 → 700
  septoria_leaf_spot: 3621 → 700
  spider_mites_two_spotted_spider_mite: 2182 → 700
  target_spot: 2284 → 700
  tomato_mosaic_virus: 2736 → 700
  tomato_yellow_leaf_curl_virus: 2534 → 700


In [29]:
random.seed(42)

SOURCE_DATASET = r"data\combined"
BALANCED_DATASET = r"data\balanced_500"

# target per tanaman
TARGETS = {
    "corn": 500,
    "mango": 500,
    "potato": 500,
    "tomato": 500
}

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")

for plant in os.listdir(SOURCE_DATASET):

    plant_path = os.path.join(SOURCE_DATASET, plant)

    if not os.path.isdir(plant_path):
        continue

    target_count = TARGETS.get(plant)

    print(f"\nProcessing {plant} (target = {target_count})")

    for disease in os.listdir(plant_path):

        disease_path = os.path.join(plant_path, disease)

        if not os.path.isdir(disease_path):
            continue

        output_folder = os.path.join(
            BALANCED_DATASET,
            plant,
            disease
        )

        os.makedirs(output_folder, exist_ok=True)
        
        files = [
            f for f in os.listdir(disease_path)
            if f.lower().endswith(VALID_EXTENSIONS)
        ]

        files.sort()

        original_count = len(files)

        # kalau lebih besar maka akan random sampling
        if original_count > target_count:

            selected_files = random.sample(files, target_count)

        # kalau lebih kecil copy semua
        else:

            selected_files = files

        # copy file
        for file in selected_files:

            src = os.path.join(disease_path, file)
            dst = os.path.join(output_folder, file)

            shutil.copy(src, dst)

        print(
            f"  {disease}: "
            f"{original_count} → {len(selected_files)}"
        )


Processing corn (target = 500)
  blight: 2146 → 500
  healthy: 2162 → 500
  leaf_spot: 998 → 500
  rust: 2300 → 500

Processing mango (target = 500)
  anthrecnose: 810 → 500
  dieback: 972 → 500
  gall_mildge_damage: 621 → 500
  healthy: 1383 → 500
  insect_damage_webber: 981 → 500
  leaf_blight: 740 → 500

Processing potato (target = 500)
  bacteria: 569 → 500
  fungi: 743 → 500
  healthy: 200 → 200
  pest: 603 → 500
  phytopthora: 309 → 309
  virus: 530 → 500

Processing tomato (target = 500)
  bacterial_spot: 3558 → 500
  early_blight: 2746 → 500
  healthy: 3706 → 500
  late_blight: 3732 → 500
  leaf_mold: 3094 → 500
  powdery_mildew: 1256 → 500
  septoria_leaf_spot: 3621 → 500
  spider_mites_two_spotted_spider_mite: 2182 → 500
  target_spot: 2284 → 500
  tomato_mosaic_virus: 2736 → 500
  tomato_yellow_leaf_curl_virus: 2534 → 500
